# Multi-Head Attention

AI & ML Foundations: Attention and Transformers · *Mathematics for AI and Machine Learning* by Fuheng Wu

Split the model width into heads: each head attends with its own projections and finds a different pattern.

Companion code for the [interactive demo](https://math4ai.org/companion/ml/multi-head/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import math

import torch

torch.manual_seed(5)

# The demo's setup: 6 tokens, model width 16, H = 4 heads of width 4
T, D, H = 6, 16, 4
dk = D // H
X = torch.randn(T, D)
mha = torch.nn.MultiheadAttention(D, H, bias=False, batch_first=True)

# ── Multi-head attention by hand, with nn.MultiheadAttention's own weights ──
Wq, Wk, Wv = mha.in_proj_weight.split(D)          # each D x D
Wo = mha.out_proj.weight


def split(t):                                     # (T, D) -> (H, T, dk)
    return t.view(T, H, dk).transpose(0, 1)


Q, K, V = split(X @ Wq.T), split(X @ Wk.T), split(X @ Wv.T)
A = torch.softmax(Q @ K.transpose(-1, -2) / math.sqrt(dk), dim=-1)   # one T x T map per head
heads = A @ V                                     # (H, T, dk)
out = heads.transpose(0, 1).reshape(T, D) @ Wo.T  # concatenate heads, mix with W_O

ref, ref_w = mha(X[None], X[None], X[None], average_attn_weights=False)
print("matches nn.MultiheadAttention:", torch.allclose(out, ref[0], atol=1e-5),
      "| same per-head maps:", torch.allclose(A, ref_w[0], atol=1e-5))

# ── Each head looks somewhere else ──────────────────────────────────────────
ent = -(A * A.log()).sum(-1).mean(-1)
print("\naverage entropy per head (log 6 = %.3f is uniform):" % math.log(T), [round(e, 3) for e in ent.tolist()])
tv = [0.5 * (A[a] - A[b]).abs().sum(-1).mean().item() for a in range(H) for b in range(a + 1, H)]
print("mean difference between heads' maps (total variation):", round(sum(tv) / len(tv), 3))

# ── The parameter count does not depend on H ────────────────────────────────
for h in (1, 2, 4, 8):
    m = torch.nn.MultiheadAttention(D, h, bias=False)
    print(f"H = {h}: {sum(p.numel() for p in m.parameters())} parameters (head width {D // h})")

matches nn.MultiheadAttention: True | same per-head maps: True

average entropy per head (log 6 = 1.792 is uniform): [1.762, 1.634, 1.569, 1.669]
mean difference between heads' maps (total variation): 0.28
H = 1: 1024 parameters (head width 16)
H = 2: 1024 parameters (head width 8)
H = 4: 1024 parameters (head width 4)
H = 8: 1024 parameters (head width 2)
